# Visualization Dashboard - AI Food Recommender

Notebook này dùng để trực quan hóa dữ liệu và kết quả cho project AI Food Recommender.

Các biểu đồ chính:
- Tổng quan dataset món ăn
- Top món phổ biến theo likes/bookmarks
- Phân bố popularity, số nguyên liệu, số bước nấu
- Thống kê cuisine
- Thống kê user profile
- Feedback like/view/dislike nếu đã có `feedback_logs.jsonl`
- Demo breakdown điểm recommendation

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.unicode_minus"] = False

DATA_PATH = Path("clean_recipe_dataset.csv")
PROFILE_DIR = Path("user_profiles")
FEEDBACK_PATH = Path("feedback_logs.jsonl")

## 1. Load dữ liệu

In [ ]:
df = pd.read_csv(DATA_PATH)

numeric_cols = ["likes", "bookmarks", "comment_count", "popularity_score", "cook_steps", "ingredient_count"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

print("Số món ăn:", len(df))
print("Số cột:", len(df.columns))
df.head()

## 2. Tổng quan dataset

In [ ]:
summary = pd.DataFrame({
    "metric": ["recipes", "avg_likes", "avg_bookmarks", "avg_cook_steps", "avg_ingredient_count"],
    "value": [
        len(df),
        df["likes"].mean(),
        df["bookmarks"].mean(),
        df["cook_steps"].mean(),
        df["ingredient_count"].mean(),
    ]
})

sns.barplot(data=summary, x="metric", y="value", color="#4C78A8")
plt.title("Tổng quan dữ liệu món ăn")
plt.xticks(rotation=20)
plt.show()

summary

## 3. Top món phổ biến theo lượt thích

In [ ]:
top_likes = df.sort_values("likes", ascending=False).head(10).copy()
top_likes = top_likes.sort_values("likes", ascending=True)

sns.barplot(data=top_likes, x="likes", y="title", color="#59A14F")
plt.title("Top 10 món có nhiều lượt thích nhất")
plt.xlabel("Lượt thích")
plt.ylabel("Món ăn")
plt.show()

top_likes[["title", "likes", "bookmarks", "popularity_score"]].sort_values("likes", ascending=False)

## 4. Top món theo bookmarks

In [ ]:
top_bookmarks = df.sort_values("bookmarks", ascending=False).head(10).copy()
top_bookmarks = top_bookmarks.sort_values("bookmarks", ascending=True)

sns.barplot(data=top_bookmarks, x="bookmarks", y="title", color="#F28E2B")
plt.title("Top 10 món được lưu nhiều nhất")
plt.xlabel("Bookmarks")
plt.ylabel("Món ăn")
plt.show()

top_bookmarks[["title", "likes", "bookmarks", "popularity_score"]].sort_values("bookmarks", ascending=False)

## 5. Phân bố popularity, số nguyên liệu và số bước nấu

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df["popularity_score"], bins=30, ax=axes[0], color="#4C78A8")
axes[0].set_title("Phân bố popularity_score")

sns.histplot(df["ingredient_count"], bins=20, ax=axes[1], color="#59A14F")
axes[1].set_title("Phân bố số nguyên liệu")

sns.histplot(df["cook_steps"], bins=20, ax=axes[2], color="#E15759")
axes[2].set_title("Phân bố số bước nấu")

plt.tight_layout()
plt.show()

## 6. Cuisine phổ biến

In [ ]:
cuisine_counts = df["cuisine"].fillna("Unknown").value_counts().head(10).reset_index()
cuisine_counts.columns = ["cuisine", "count"]

sns.barplot(data=cuisine_counts, x="count", y="cuisine", color="#B07AA1")
plt.title("Top cuisine trong dataset")
plt.xlabel("Số món")
plt.ylabel("Cuisine")
plt.show()

cuisine_counts

## 7. User profiles

In [ ]:
profiles = []

if PROFILE_DIR.exists():
    for path in PROFILE_DIR.glob("*.json"):
        with open(path, "r", encoding="utf-8") as f:
            profile = json.load(f)
        features = profile.get("features", {})
        profiles.append({
            "user_id": profile.get("user_id"),
            "age": profile.get("age"),
            "gender": profile.get("gender"),
            "weight": profile.get("weight"),
            "height": profile.get("height"),
            "goal": profile.get("goal"),
            "bmi": features.get("bmi"),
            "bmi_label": features.get("bmi_label"),
        })

profiles_df = pd.DataFrame(profiles)
profiles_df

In [ ]:
if not profiles_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.barplot(data=profiles_df, x="user_id", y="bmi", ax=axes[0], color="#4C78A8")
    axes[0].set_title("BMI theo user")
    axes[0].set_xlabel("User")
    axes[0].set_ylabel("BMI")

    goal_counts = profiles_df["goal"].fillna("Unknown").value_counts().reset_index()
    goal_counts.columns = ["goal", "count"]
    sns.barplot(data=goal_counts, x="goal", y="count", ax=axes[1], color="#F28E2B")
    axes[1].set_title("Số user theo mục tiêu")
    axes[1].set_xlabel("Goal")
    axes[1].set_ylabel("Số user")

    plt.tight_layout()
    plt.show()
else:
    print("Chưa có profile để trực quan hóa.")

## 8. Feedback like/view/dislike

In [ ]:
feedback_logs = []

if FEEDBACK_PATH.exists():
    with open(FEEDBACK_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                feedback_logs.append(json.loads(line))

feedback_df = pd.DataFrame(feedback_logs)
feedback_df.head()

In [ ]:
if not feedback_df.empty:
    action_counts = feedback_df["action"].value_counts().reset_index()
    action_counts.columns = ["action", "count"]

    sns.barplot(data=action_counts, x="action", y="count", palette="Set2", hue="action", legend=False)
    plt.title("Số lượng feedback theo hành động")
    plt.xlabel("Action")
    plt.ylabel("Số lượt")
    plt.show()

    display(action_counts)
else:
    print("Chưa có feedback_logs.jsonl. Hãy bấm Thích / Đã xem / Không thích trong app để tạo dữ liệu.")

## 9. Demo breakdown điểm recommendation

Biểu đồ này minh họa cách giải thích điểm cho một món ăn. Khi chạy pipeline thật, các điểm này đến từ `Ranking_Engine_08.py`.

In [ ]:
demo_breakdown = pd.DataFrame({
    "component": [
        "similarity",
        "keyword",
        "title_keyword",
        "goal",
        "ingredient",
        "cuisine",
        "popularity",
    ],
    "score": [0.92, 1.00, 1.00, 0.50, 1.00, 1.00, 0.25],
})

sns.barplot(data=demo_breakdown, x="score", y="component", color="#4C78A8")
plt.title("Demo breakdown điểm cho query 'cơm'")
plt.xlabel("Điểm")
plt.ylabel("Thành phần")
plt.xlim(0, 1)
plt.show()

demo_breakdown

## 10. Demo sustainability theo ngày

Nếu chưa có meal plan từ app, cell này tạo dữ liệu demo để thuyết trình.

In [ ]:
demo_sustainability = pd.DataFrame({
    "day": [f"Day {i}" for i in range(1, 8)],
    "sustainability_score": [0.65, 0.72, 0.58, 0.80, 0.69, 0.74, 0.61],
})

sns.lineplot(data=demo_sustainability, x="day", y="sustainability_score", marker="o", color="#59A14F")
plt.title("Demo sustainability score theo ngày")
plt.xlabel("Ngày")
plt.ylabel("Sustainability score")
plt.ylim(0, 1)
plt.show()

demo_sustainability